# Local AI Model: Training and Troubleshooting Report
**By: Sean Yoo**

## Overview

The goal of this project was to take a small AI model called **Qwen3-0.6B** and teach it about a custom dataset containing airport information. I used **LoRA** (Low-Rank Adaptation), a method that lets you customize an existing model without having to retrain the entire model from scratch.

The model was trained locally on an Apple Silicon Mac using Apple's MLX tools. After training, I also tested whether the customized model could be loaded and used to generate responses.

The actual AI training worked. Most of the difficulty came from getting the different software libraries on my computer to work together correctly.

## Problems I Ran Into

### 1. Different MPI Versions Were Conflicting

The first major problem came from a piece of software called **MPI** (Message Passing Interface), which helps programs communicate with each other. Even though I was only training the model on one computer, the MLX training program still checked for MPI.

My computer had two different MPI installations: one that came with Anaconda and another installed through Homebrew. MLX needed the Homebrew version, **Open MPI**, but it was finding the Anaconda version, **MPICH** (Message Passing Interface Chameleon), first. Because MLX could not use that version, the training process stopped before it could save the trained model.

This also explained why the model's adapter folder was missing the important `adapters.safetensors` file. The training had crashed before it reached the point where that file was created.

### 2. Editing the Installed MLX Files Was Not a Good Long-Term Fix

I initially tried changing the MLX source code installed on my computer to force it to use the correct MPI setup. This worked around part of the problem, but it was fragile because those changes could disappear if the package were updated.

It also did not completely solve the issue because MLX remembers which communication method it starts with when a program begins. Changing individual parts of the code therefore did not reliably control the final setup.

### 3. The Missing Model File Caused Another Error

When I tried to generate text using the trained adapter, the program reported that it could not find `adapters.safetensors`. At first this looked like a separate problem, but it was actually caused by the earlier training crash.

Once training was able to finish properly, this error went away.

### 4. Another Approach Created a Hostfile Error

I also tried using MLX's launcher to force the correct setup. This avoided the original MPI problem, but it created a different error involving a **hostfile**, which is a small file used to describe the computers or processes involved in a distributed run.

Since I was only running the model locally on one computer, this approach was unnecessary. I abandoned it and used a simpler solution instead.

## How I Fixed It

The successful fix did not require changing the MLX code or permanently modifying my computer. Instead, I temporarily changed which software folders the terminal looked at during the training session.

I made sure the terminal could find my Python environment and the Homebrew version of Open MPI, while keeping the Anaconda directories out of the search path. This prevented MLX from accidentally picking up the wrong MPI installation.

After making that temporary change, I was able to run the training command directly. The model trained successfully from start to finish without the earlier MPI errors.

### Environment Setup Used for the Fix


- export PATH="$PWD/.venv/bin:/opt/homebrew/bin:/usr/local/bin:/usr/bin:/bin"

- export DYLD_LIBRARY_PATH="/opt/homebrew/lib"


These changes were only needed for the training session; they did not permanently modify the system.

## Results

- Training completed successfully for **10 iterations** with no MPI errors.
- The validation loss decreased from **3.455 to 1.957**, meaning the model's performance improved during the test run.
- The `adapters.safetensors` file was successfully created and was about **2.8 MB**, confirming that the trained adapter was saved correctly.
- The customized model could be loaded successfully and used to generate text.
- The main problem was traced back to a conflict between Anaconda's MPICH installation and Homebrew's Open MPI installation.
- The final solution did not require permanent changes to the computer or modifications to the installed MLX packages.

## What I Would Do Next

1. **Run many more training iterations.** The 10-iteration run was mainly a quick test to make sure the entire system worked. A real training run would use roughly 200–1,000 or more iterations.
2. **Experiment with the training settings.** I would increase the number of model layers used during training and try a larger batch size to give the model more opportunity to learn.
3. **Generate longer responses.** I would increase the maximum number of tokens generated by the model because the current default of 100 tokens can cause longer answers to be cut off.
4. **Automate the environment setup.** I could create a small shell script or use a tool such as `direnv` to automatically set up the correct environment before training instead of typing the commands manually each time.

## Takeaway

The biggest lesson from this project was that getting a local AI model to train is as much about the surrounding software environment as it is about the model or the training data themselves.

In this case, the model and training setup were working, but two different versions of a supporting library were conflicting. Once I identified that conflict and made the computer use the correct version, the training process worked as expected.

This gave me a working foundation that I can now build on with a larger training run and a more useful dataset.